# 🔐 Certificate Classification - Part 1: Data Exploration

## Project Goal
Build a machine learning pipeline to **classify TLS/SSL certificates as malicious or benign** based on certificate metadata.

### Why This Matters
- Malicious actors use SSL certificates for phishing, malware C2, and fraud
- Early detection of suspicious certificates can prevent attacks
- Automating classification scales security operations

### Data Sources
- **CRT.sh**: Certificate Transparency logs (~948K certificates)
- **MalwareBazaar**: Known malicious domain indicators (~123K domains)
- **SSLBL**: Abuse.ch SSL Blacklist

### Labels
- `malicious`: Certificate linked to known bad domains/indicators
- `unknown`: No match found (treat as presumed benign for training)

---

## 1. Setup & Imports

In [ ]:
# Core
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_style('whitegrid')
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print('Seaborn not installed - using matplotlib only')

# Paths
DATA_PATH = Path('./outputs/ml/labeled_sheet.csv')
print(f'Data file: {DATA_PATH}')
print(f'File size: {DATA_PATH.stat().st_size / 1e6:.1f} MB')

## 2. Load Data

In [ ]:
# Load the labeled dataset
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f'Total records: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Column info
df.info()

## 3. Label Distribution (Class Imbalance)

In [ ]:
# Check label distribution
label_counts = df['label'].value_counts()
print('Label Distribution:')
print(label_counts)
print(f'\nImbalance ratio: 1:{label_counts["unknown"] // max(label_counts.get("malicious", 1), 1)}')

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'crimson'])
axes[0].set_title('Label Distribution (Count)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Log scale
label_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'crimson'], logy=True)
axes[1].set_title('Label Distribution (Log Scale)')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\n⚠️ SEVERE CLASS IMBALANCE - Will need special handling!')

## 4. Feature Analysis

In [ ]:
# Analyze each column
print('=== Column Analysis ===\n')

for col in df.columns:
    print(f'📊 {col}')
    print(f'   Type: {df[col].dtype}')
    print(f'   Non-null: {df[col].notna().sum():,} ({df[col].notna().mean()*100:.1f}%)')
    print(f'   Unique: {df[col].nunique():,}')
    if df[col].dtype == 'object':
        print(f'   Top 3: {df[col].value_counts().head(3).to_dict()}')
    elif df[col].dtype in ['int64', 'float64']:
        print(f'   Range: {df[col].min()} - {df[col].max()}')
    print()

## 5. Signature Algorithm Analysis

In [ ]:
# Signature hash algorithms
print('Signature Hash Algorithms:')
print(df['signature_hash_algo'].value_counts())

In [ ]:
# Compare signature algo by label
sig_by_label = pd.crosstab(df['signature_hash_algo'], df['label'], normalize='columns') * 100
print('\nSignature Algorithm Distribution by Label (%):')
print(sig_by_label.round(2))

## 6. Key Size Analysis

In [ ]:
# Convert public_key_size to numeric
df['public_key_size_num'] = pd.to_numeric(df['public_key_size'], errors='coerce')

print('Public Key Size Distribution:')
print(df['public_key_size_num'].value_counts().head(10))

In [ ]:
# Key size by label
print('\nKey Size by Label:')
print(df.groupby('label')['public_key_size_num'].describe())

## 7. SAN Count Analysis

In [ ]:
# SAN count distribution
print('SAN Count Statistics:')
print(df.groupby('label')['san_count'].describe())

In [ ]:
# Visualize SAN count
fig, ax = plt.subplots(figsize=(10, 4))
df[df['san_count'] <= 20]['san_count'].hist(bins=20, ax=ax, alpha=0.7)
ax.set_xlabel('SAN Count')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of SAN Count (≤20)')
plt.show()

## 8. Matched Domains (Malicious Certificates)

In [ ]:
# Examine malicious certificates
malicious = df[df['label'] == 'malicious'].copy()
print(f'Total malicious certificates: {len(malicious)}')
print(f'\nTop matched domains:')
print(malicious['matched_domain'].value_counts().head(15))

In [ ]:
# Reasons for malicious label
print('\nReasons breakdown:')
all_reasons = malicious['reasons'].dropna().str.split('|').explode()
reason_counts = all_reasons.value_counts()
print(reason_counts.head(10))

## 9. Certificate Validity Period

In [ ]:
# Parse dates
df['not_before_dt'] = pd.to_datetime(df['not_before'], errors='coerce')
df['not_after_dt'] = pd.to_datetime(df['not_after'], errors='coerce')

# Calculate validity period in days
df['validity_days'] = (df['not_after_dt'] - df['not_before_dt']).dt.days

print('Validity Period (days) by Label:')
print(df.groupby('label')['validity_days'].describe())

## 10. Key Findings Summary

In [ ]:
print('='*60)
print('📋 KEY FINDINGS')
print('='*60)
print(f'''
1. DATASET SIZE
   - Total certificates: {len(df):,}
   - Malicious: {len(malicious):,} ({len(malicious)/len(df)*100:.3f}%)
   - Unknown (presumed benign): {len(df) - len(malicious):,}

2. CLASS IMBALANCE
   - Ratio: ~1:{len(df)//max(len(malicious),1)} (extreme imbalance!)
   - Strategy needed: SMOTE, class weights, or undersampling

3. USEFUL FEATURES
   - signature_hash_algo: SHA-1 vs SHA-256 matters
   - public_key_size: 1024-bit keys are weak
   - san_count: Number of Subject Alternative Names
   - validity_days: Certificate lifetime
   - common_name: Domain text features

4. WEAK CRYPTO INDICATORS
   - SHA-1 signatures (deprecated)
   - RSA-1024 keys (weak)

5. NEXT STEPS
   → Feature engineering (02_feature_engineering.ipynb)
   → Model training (03_model_training.ipynb)
   → Model comparison (04_model_comparison.ipynb)
''')

In [ ]:
# Save processed dataframe for next notebook
df.to_pickle('./outputs/ml/df_explored.pkl')
print('✅ Saved explored dataframe to df_explored.pkl')